---
title: "Chương 5: Hiệu năng"
short_title: "5. Hiệu năng"
---

## 1. Phương pháp đo

- **Môi trường:** Google Colab, runtime **CPU-only** (không GPU), cùng một bộ code cho mọi khoảng ID.
- **Cách chạy:** `chunked_crawl.py` crawl **lần lượt** khoảng ID của từng thành viên trên cùng một runtime, mỗi lúc một crawler
  (giới hạn tốc độ của arXiv tính cho mọi máy cộng lại). Mỗi khoảng chia thành các bước (discovery, từng đoạn 1000 bài,
  ngày nộp từ Kaggle, sweep); mỗi bước chạy dưới `benchmark.py` và được lưu lên Drive khi xong, nên khi Colab ngắt
  (giới hạn 12 giờ) runtime mới khôi phục và làm tiếp từ bước dở.
- **Thời gian:** wall time của mỗi lần chạy (invocation) đo bằng `time.perf_counter()` từ lúc khởi động đến lúc crawler kết thúc, gồm cả thời gian chờ giới hạn request và các lượt retry. Mỗi bước là một invocation; thời gian của một khoảng là **tổng các invocation đã kết thúc** (`benchmark_runs.py`), của cả nhóm là tổng mọi khoảng. Không tính: thời gian lưu/khôi phục bản lưu (ghi riêng trong `snapshot_log.csv`), cài đặt môi trường và khoảng nghỉ giữa các phiên Colab; một bước bị ngắt giữa chừng được liệt kê trong cảnh báo và chạy lại ở phiên sau.
- **RAM:** RSS của **cả cây tiến trình** crawler (tiến trình chính và mọi tiến trình con), lấy mẫu mỗi 2 giây bằng `psutil`. RAM tối đa là mẫu lớn nhất, RAM trung bình là trung bình các mẫu.
- **Dung lượng đĩa:** dung lượng thư mục dữ liệu, quét mỗi 300 giây (tương đương `du`); dung lượng tối đa là mẫu lớn nhất, dung lượng đầu ra là lần quét cuối khi crawler kết thúc.
- **Cả nhóm:** thời gian, số bài và dung lượng đầu ra được cộng lại; RAM tối đa lấy giá trị lớn nhất (mỗi lúc chỉ một crawler chạy). **Dung lượng đĩa tối đa** là lúc các thư mục trên ổ cộng lại lớn nhất, vì thư mục của khoảng đã xong vẫn nằm trên ổ của runtime (`group_benchmark.py --one-machine`).

In [ ]:
import pandas as pd
import plotly.express as px
from report_utils import (COLORS, FINAL_SOURCE, REF_OK, group, human_bytes, human_seconds,
                          load, members, meta, metric, mib, one_machine, pct, provenance, show, table)
provenance()

In [ ]:
m, g = members(), group()
stages = load("run_log")
stages["seconds"] = pd.to_numeric(stages["seconds"], errors="coerce")
discovery = stages[stages["stage"] == "discovery"].groupby("member")["seconds"].sum()

def row(r, disc):
    return {"Thành viên": r["member"], "Số bài": f"{r['papers']:,}", "Số lần chạy": r["invocations"],
            "Tổng wall time": human_seconds(r["wall_seconds"]),
            "Thời gian TB mỗi bài": human_seconds(r["seconds_per_paper"]),
            "Thời gian discovery": human_seconds(disc),
            "RAM tối đa": human_bytes(r["ram_peak_bytes"]), "RAM trung bình": human_bytes(r["ram_mean_bytes"]),
            "Đĩa tối đa": human_bytes(r["data_dir_peak_bytes"]),
            "Dung lượng đầu ra": human_bytes(r["data_dir_final_bytes"])}

rows = [row(r, discovery.get(r["member"])) for _, r in m.iterrows()]
rows.append(row(g, discovery.sum()))
rows[-1]["Thành viên"] = "Cả nhóm"
table(pd.DataFrame(rows), "Bảng 5.1. Thời gian chạy và bộ nhớ theo thành viên")
clock = "wall_seconds" if m["wall_seconds"].notna().all() else "crawler_seconds"
longest = m[clock].max()
if one_machine():
    print(f"Các khoảng chạy lần lượt trên cùng một runtime: crawl dữ liệu của cả nhóm mất "
          f"{human_seconds(g[clock])} (khoảng lâu nhất: {human_seconds(longest)}).")
else:
    print(f"Các thành viên chạy song song trên các runtime riêng: dữ liệu của cả nhóm hoàn tất sau "
          f"{human_seconds(longest)} (thành viên chạy lâu nhất); tổng thời gian máy của cả nhóm là "
          f"{human_seconds(g[clock])}.")
if clock == "crawler_seconds":
    print("Lưu ý: có thành viên không có số đo của benchmark.py, nên thời gian so sánh theo đồng hồ của main.py.")

"Thời gian TB mỗi bài" bằng wall time chia cho số bài tồn tại: đây là **thông lượng** của cả pipeline, gồm cả thời gian chờ giới hạn request của arXiv và Semantic Scholar, không phải độ trễ khi xử lý riêng một bài.

## 2. Thời gian từng bước

Thời gian các bước ghi trong `run_log.csv`, cộng qua mọi lần chạy. Bước tải mã nguồn và bước references chạy song song nên tổng các bước lớn hơn wall time.

In [ ]:
by_stage = stages.groupby(["member", "stage"])["seconds"].sum().reset_index()
by_stage["giờ"] = by_stage["seconds"] / 3600
fig = px.bar(by_stage, x="member", y="giờ", color="stage", barmode="group", color_discrete_sequence=COLORS,
             title="Tổng thời gian từng bước theo thành viên",
             labels={"member": "Thành viên", "giờ": "Giờ", "stage": "Bước"})
show(fig)

## 3. Thời gian xử lý mã nguồn của một bài

Tổng thời gian tải và trích xuất các phiên bản của một bài (`total_seconds` trong `versions.json`), không gồm thời gian chờ trong hàng đợi giới hạn request:

In [ ]:
versions = load("versions")
per_paper = (versions[versions["status"].isin(FINAL_SOURCE)]
             .groupby(["member", "arxiv_id"])["total_seconds"].sum().reset_index())
fig = px.box(per_paper, x="member", y="total_seconds", log_y=True, color_discrete_sequence=COLORS,
             title="Thời gian tải + trích xuất mã nguồn mỗi bài",
             labels={"member": "Thành viên", "total_seconds": "Giây (thang log)"})
show(fig)
d = per_paper["total_seconds"]
table(pd.DataFrame({"Thống kê": ["Số bài", "Trung bình", "Trung vị", "Phân vị 95", "Lớn nhất"],
                    "Giá trị": [f"{len(d):,}", human_seconds(d.mean()), human_seconds(d.median()),
                                human_seconds(d.quantile(0.95)), human_seconds(d.max())]}),
      "Bảng 5.2. Thời gian xử lý mã nguồn mỗi bài")

## 3b. Thời gian xử lý trọn vẹn một bài (chạy riêng)

Thời gian TB mỗi bài ở Bảng 5.1 là thông lượng của lượt chạy lớn. Để có thời gian xử lý **một** bài từ đầu đến cuối
(discovery, metadata, tải và trích xuất mọi phiên bản, references), một mẫu ngẫu nhiên các bài được chạy lại riêng từng bài
trên thư mục mới (`per_paper_benchmark.py`), ngoài lượt chạy chính.

In [ ]:
if meta().get("paper_timings"):
    pt = load("paper_timings")
    pt["wall_seconds"] = pd.to_numeric(pt["wall_seconds"], errors="coerce")
    ok = pt[pt["status"] == "complete"]["wall_seconds"]
    table(pd.DataFrame({"Thống kê": ["Số bài trong mẫu", "Số bài hoàn tất", "Trung bình", "Trung vị", "Nhỏ nhất", "Lớn nhất"],
                        "Giá trị": [f"{len(pt):,}", f"{len(ok):,}", human_seconds(ok.mean()), human_seconds(ok.median()),
                                    human_seconds(ok.min()), human_seconds(ok.max())]}),
          "Bảng 5.2b. Thời gian xử lý trọn vẹn một bài, chạy riêng từng bài")
else:
    print("Chưa có số liệu: chạy Bước 8b của notebook crawler rồi xuất lại data/m1.")

## 4. RAM và dung lượng đĩa theo thời gian

Mỗi đường là một lần chạy `benchmark.py` (đã lược bớt mẫu khi xuất dữ liệu).

In [ ]:
samples = load("performance_samples")
for col in ["elapsed_seconds", "rss_tree_bytes", "data_dir_bytes"]:
    samples[col] = pd.to_numeric(samples[col], errors="coerce")
samples["giờ"] = samples["elapsed_seconds"] / 3600
samples["RAM (MB)"] = samples["rss_tree_bytes"].map(mib)
samples["lần chạy"] = samples["member"] + " · " + samples["run_id"].astype(str)
fig = px.line(samples, x="giờ", y="RAM (MB)", color="lần chạy", color_discrete_sequence=COLORS,
              title="RAM của cây tiến trình crawler theo thời gian",
              labels={"giờ": "Thời gian từ lúc bắt đầu lần chạy (giờ)"})
show(fig)
disk = samples.dropna(subset=["data_dir_bytes"]).copy()
disk["Thư mục dữ liệu (GB)"] = disk["data_dir_bytes"] / 2**30
fig = px.line(disk, x="giờ", y="Thư mục dữ liệu (GB)", color="lần chạy", markers=True,
              color_discrete_sequence=COLORS, title="Dung lượng thư mục dữ liệu theo thời gian",
              labels={"giờ": "Thời gian từ lúc bắt đầu lần chạy (giờ)"})
show(fig)

## 5. Các lần chạy và giới hạn request của arXiv

In [ ]:
runs = load("benchmark_runs")
runs_view = pd.DataFrame({
    "Thành viên": runs["member"], "run_id": runs["run_id"], "Mã thoát": runs["return_code"],
    "Bị ngắt": runs["interrupted"], "Wall time": runs["wall_seconds"].map(human_seconds),
    "RAM tối đa": runs["ram_sampled_peak_bytes"].map(human_bytes),
    "Đĩa tối đa": runs["data_dir_sampled_peak_bytes"].map(human_bytes)})
table(runs_view, "Bảng 5.3. Từng lần chạy benchmark.py")
req = pd.DataFrame({
    "Thành viên": m["member"], "Request arXiv API": m["arxiv_api_requests"],
    "Request tải nguồn": m["arxiv_download_requests"],
    "Thời gian tối thiểu theo nhịp arXiv": m["arxiv_floor_seconds"].map(human_seconds),
    "Thời gian bị arXiv chặn (tạm dừng)": m["arxiv_pause_seconds"].map(human_seconds),
    "Số lần tạm dừng": m["arxiv_pauses"]})
table(req, "Bảng 5.4. Số request gửi tới arXiv")

"Thời gian tối thiểu theo nhịp arXiv" là thời gian mà chính số request đã gửi cần nếu gửi đúng nhịp arXiv yêu cầu (3 giây mỗi lời gọi API, trung bình 0,5 giây mỗi request tải nguồn). Phần lớn wall time là thời gian chờ giới hạn request và tạm dừng khi arXiv từ chối, không phải thời gian tính toán; vì vậy RAM của crawler thấp và gần như không đổi trong suốt quá trình chạy.